# Stage 8: Deconvolution of MoTrPAC Bulk RNA-seq into Pseudo-cells

**Pipeline script:** `pipeline/run_stage8.py` (all tissues: `pipeline/run_deconv_all.py`)  
**Outputs read:** the active config's `deconvolution.results_dir` (printed by the first code cell). Since 2026-11-03 the
default is the clean 16-tissue rebuild (jobs 12121714–12121729, 2026-11-02), which reproduces the earlier assembled
tree byte for byte (`deconvolution/REFS4ALL_REBUILD_RESULTS.md`). References: the jointly re-annotated `references_v4/`
plus the BLOOD whole-blood composite; BAT, HEART, HYPOTH and TESTES use `references_v3/`. Each tissue's reference is
listed in `deconvolution/tissue_references.yaml`.

Stage 8 converts each MoTrPAC tissue's bulk RNA-seq profiles into cell-type-resolved posterior expression profiles.
Each tissue is processed independently through identifier harmonization, BayesPrism deconvolution, posterior-Z
extraction, and pseudo-cell construction. The table below gives, per tissue, the reference, its cell types, the bulk
samples and the pseudo-cells (TESTES is male-only and OVARY female-only, so they have fewer samples).

> **Model independence.** Stage 8 runs before GeneCompass and does not use the model, so its outputs are the
> same for retrain v2 (the working model) and April's model. The Stage 9 folders
> (`<genecompass_input_dir>/<tissue>/pseudocells.h5ad`) hold these pseudo-cells. The 14-tissue refs4
> version of this notebook is in git history at commit `542b454` (July references: `1281d21`); the comparisons are
> `deconvolution/ALL_TISSUE_RERUN_RESULTS.md` (refs4 vs all tissues) and `deconvolution/REFS4_RERUN_RESULTS.md` (July vs refs4).

```
MoTrPAC bulk counts (Rnor_6.0 Ensembl IDs, ~release 96)
    -> current rat Ensembl identifiers (the BioMart release in the config)
    -> BayesPrism with a tissue-matched single-cell reference
    -> posterior expression by sample and cell type
    -> one pseudo-cell per sample x cell type
```

Three MoTrPAC bulk tissues are not deconvolved. VENACV was built and then dropped (the only proxy reference is
lung-contaminated); ADRNL has no rat reference; COLON's only candidate (sorted cells without epithelium) was
rejected (`deconvolution/REFERENCE_RELABEL_PREREG.md` §14). SPLEEN and OVARY were added on 2026-11-01 and are read
differentially only: SPLEEN's estimated fractions are implausible, and the OVARY reference is one animal of unknown
strain. The table below is read from `deconvolution/tissue_references.yaml` and the files on disk.

In [1]:
import pandas as pd, yaml, anndata as ad
from pathlib import Path

from nb_config import PROJECT, RES_DIR, GC_DIR, TISSUES, prism_minutes
REFMAP  = yaml.safe_load(open(PROJECT / "deconvolution/tissue_references.yaml"))

rows = []
for t in TISSUES:
    ref = PROJECT / REFMAP[t]["reference_dir"]
    meta = pd.read_csv(ref / "cells_meta.tsv", sep="\t", usecols=["cell_type"])
    pz = RES_DIR / t / "pred_z"
    z_types = [l for l in (pz / "types.txt").read_text().splitlines() if l.strip()]
    n_genes = sum(1 for l in (pz / "genes.txt").read_text().splitlines() if l.strip())
    pc = ad.read_h5ad(GC_DIR / t.lower() / "pseudocells.h5ad", backed="r")
    rows.append({"tissue": t, "reference": ref.name, "ref cells": len(meta),
                 "ref types": meta.cell_type.nunique(), "Z types": len(z_types),
                 "types match": set(z_types) == set(meta.cell_type.unique()),
                 "Z genes": n_genes, "samples": pc.obs["sample"].nunique(),
                 "pseudo-cells": pc.n_obs, "pc genes": pc.n_vars,
                 "deconvolved": pd.Timestamp((RES_DIR / t / "bp_result.rds").stat().st_mtime, unit="s").strftime("%Y-%m-%d")})
    pc.file.close()
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print(f"\n{len(inv)} tissues, {inv['pseudo-cells'].sum():,} pseudo-cells; "
      f"Z cell types equal the reference's in {inv['types match'].sum()} of {len(inv)} tissues")

config: (default config)
Stage 8: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/results_refs4all_rebuild/motrpac
Stage 9: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/genecompass_input_v2_refs4all_rebuild
16 tissues: BAT, BLOOD, CORTEX, HEART, HIPPOC, HYPOTH, KIDNEY, LIVER, LUNG, OVARY, SKMGN, SKMVL, SMLINT, SPLEEN, TESTES, WATSC


tissue             reference  ref cells  ref types  Z types  types match  Z genes  samples  pseudo-cells  pc genes deconvolved
   BAT         BAT_GSE244451      28246          6        6         True    14162       50           300     14162  2026-11-02
 BLOOD       BLOOD_composite      67147         13       13         True    14920       50           650     14920  2026-11-02
CORTEX      CORTEX_GSE303115      12936         12       12         True    15385       50           600     15385  2026-11-02
 HEART    HEART_GSE280111_LV     135288         16       16         True    15864       50           800     15864  2026-11-02
HIPPOC      HIPPOC_GSE295314      23892         11       11         True    15313       50           550     15313  2026-11-02
HYPOTH    HYPOTH_GSE248413_Y       8471         13       13         True    14061       50           650     14061  2026-11-02
KIDNEY      KIDNEY_GSE240658      24479         15       15         True    14892       50           750     14

## Steps 1–2 — Bulk Liftover & BayesPrism Deconvolution

`deconvolution/R/prepare_motrpac_bulk.sh` lifts each `TRNSCRPT_<TISSUE>_RAW_COUNTS.rda` from its Rnor_6.0 Ensembl IDs
to current ENSRNOG identifiers, so the bulk gene space matches the GeneCompass tokenizer vocabulary
(`deconvolution/MOTRPAC_BULK_LIFTOVER.md`).

`run_deconvolution.sh` calls **BayesPrism**, which learns a prior over cell-type-specific gene expression from the
single-cell reference, then estimates per-sample cell-type fractions and the posterior expression matrix **Z** via
Gibbs sampling. Pre-processing applied to the reference before fitting (each run's settings are recorded in its
`provenance_run_deconvolution.json` and printed by the next cell):

- Sex-chromosome genes removed (`exclude_sex_chromosomes`)
- Non-protein-coding genes removed (`protein_coding_only`)
- Ribosomal, mitochondrial and haemoglobin genes removed (the curated `deconvolution/reference/rat_exclude_genes.tsv`;
  BayesPrism's own cleanup is human/mouse only)
- Reference genes expressed in fewer than 3 cells removed
- BayesPrism's outlier filter (`outlier_cut`, `outlier_fraction`)

Gene intersection (reference ∩ bulk) is handled internally by BayesPrism; the keep-list
`deconvolution/reference/rat_genecompass_genes.tsv` restricts output to the GeneCompass gene vocabulary.

In [2]:
import json
prov = {t: json.load(open(RES_DIR / t / "provenance_run_deconvolution.json")) for t in TISSUES}
cols = ["bayesprism", "n_cores", "exclude_sex_chromosomes", "protein_coding_only", "outlier_cut", "outlier_fraction",
        "n_ref_cells", "n_ref_genes", "n_mixtures"]
pv = pd.DataFrame([{"tissue": t, "R": p.get("r_version"), "run": p["created"][:10],
                    **{c: p["params"].get(c) for c in cols}} for t, p in prov.items()])
print(pv.to_string(index=False))
excl = pd.read_csv(PROJECT / "deconvolution/reference/rat_exclude_genes.tsv", sep="\t")
print(f"\nrat_exclude_genes.tsv: {len(excl)} genes (" +
      ", ".join(f"{k} {v}" for k, v in excl["category"].value_counts().items()) + ")")

tissue     R        run bayesprism  n_cores  exclude_sex_chromosomes  protein_coding_only  outlier_cut  outlier_fraction  n_ref_cells  n_ref_genes  n_mixtures
   BAT 4.4.1 2026-11-01      2.2.3       16                     True                 True         0.01               0.1        28246        14163          50
 BLOOD 4.4.1 2026-11-01      2.2.3       16                     True                 True         0.01               0.1        67147        14931          50
CORTEX 4.4.1 2026-11-01      2.2.3       16                     True                 True         0.01               0.1        12936        16030          50
 HEART 4.4.1 2026-11-02      2.2.3       16                     True                 True         0.01               0.1       135288        16086          50
HIPPOC 4.4.1 2026-11-02      2.2.3       16                     True                 True         0.01               0.1        23892        15313          50
HYPOTH 4.4.1 2026-11-01      2.2.3       16   

In [3]:
# BayesPrism run.prism wall time (min): from each run's provenance (params.run_prism_minutes); runs made
# before it was recorded read deconvolution/reference/run_prism_minutes_backfill.tsv (values from the SLURM logs)
import matplotlib.pyplot as plt, numpy as np
inv["prism_source"] = inv.tissue.map(lambda t: prism_minutes(t)[1])
inv["prism_min"] = inv.tissue.map(lambda t: prism_minutes(t)[0])

colors = plt.cm.tab20(np.linspace(0, 1, len(inv)))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle("Stage 8 — BayesPrism Deconvolution", fontsize=13, fontweight="bold")
y = np.arange(len(inv))
for ax, vals, lab, fmt in [(ax1, inv.prism_min, "BayesPrism run.prism wall time (min)", "{:.1f} min"),
                           (ax2, inv["ref cells"] / 1e3, "SC reference size (×1,000 cells)", "{:.0f}K")]:
    bars = ax.barh(y, vals, color=colors, edgecolor="white", height=0.7)
    for bar, v in zip(bars, vals):
        ax.text(v + vals.max() * 0.01, bar.get_y() + bar.get_height() / 2, fmt.format(v), va="center", fontsize=8.5)
    ax.set_yticks(y); ax.set_yticklabels(inv.tissue); ax.set_xlabel(lab); ax.set_xlim(0, vals.max() * 1.22)
    ax.invert_yaxis()
ax1.set_title("BayesPrism Run Time per Tissue"); ax2.set_title("Single-Cell Reference Size")
plt.show()

## Step 3 — Extract Z: Posterior Per-Cell-Type Expression

`extract_z.sh` calls `extract_z.R`, which retrieves the BayesPrism posterior **Z** matrix — the estimated gene expression profile for each cell type in each sample. The output `pred_z/` directory contains:

- `genes.txt` — one ENSRNOG per line (the gene space after BayesPrism filtering)
- `types.txt` — one cell type label per line
- `predz__<CellType>.csv` — `samples × genes` matrix for that cell type

These per-cell-type matrices preserve per-sample variation needed for the Aim 2 contrasts (sex × timepoint × training), since aggregating to group means would discard biological variation.


In [4]:
rows = []
for t in TISSUES:
    ctypes = [l for l in (RES_DIR / t / "pred_z" / "types.txt").read_text().splitlines() if l.strip()]
    rows.append({"Tissue": t, "Cell types": len(ctypes),
                 "Cell type labels (truncated)": ", ".join(ctypes[:5]) + (" ..." if len(ctypes) > 5 else "")})
print(pd.DataFrame(rows).to_string(index=False))

Tissue  Cell types                                                                                                          Cell type labels (truncated)
   BAT           6                                                         Brown adipocytes, Immune cells, Endothelial cells, ASPC, SMCs & Pericytes ...
 BLOOD          13                                                 CD8+ T cells, Classical monocytes, CD4+ T cells, Non-classical monocytes, B cells ...
CORTEX          12                   Oligodendrocytes, Oligodendrocyte precursor cells, Excitatory neurons, Medium spiny neurons, Inhibitory neurons ...
 HEART          16                                                        Cardiac fibroblasts, T cells, Endothelial cells, Cardiomyocytes, Pericytes ...
HIPPOC          11            Inhibitory neurons, Excitatory neurons, Dentate gyrus granule cells, Oligodendrocyte precursor cells, Oligodendrocytes ...
HYPOTH          13                                                  Glutamatergic 

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
fig.suptitle("Stage 8 — Cell Type Resolution per Tissue", fontsize=13, fontweight="bold")
x = np.arange(len(inv))
for ax, col, lab in [(ax1, "Z types", "Cell types resolved by BayesPrism"),
                     (ax2, "pseudo-cells", "Pseudo-cells (samples × cell types)")]:
    bars = ax.bar(x, inv[col], color=colors, edgecolor="white")
    for bar, v in zip(bars, inv[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, v * 1.01, f"{v:,}", ha="center", va="bottom", fontsize=8.5)
    ax.set_xticks(x); ax.set_xticklabels(inv.tissue, rotation=35, ha="right"); ax.set_ylabel(lab)
    ax.set_ylim(0, inv[col].max() * 1.15)
ax1.set_title(f"Cell Types per Tissue (mean {inv['Z types'].mean():.1f})")
ax2.set_title(f"Pseudo-cells per Tissue (total {inv['pseudo-cells'].sum():,})")
plt.show()

## Step 4 — Build Pseudo-cells (`build_pseudocells.py`)

`build_pseudocells.py` converts the `pred_z/` matrices into a single AnnData object:

| Property | Value |
|----------|-------|
| Granularity | 1 pseudo-cell per (bulk sample × cell type) |
| `obs` columns | `pseudocell_id`, `sample`, `cell_type`, `tissue` |
| `var_names` | ENSRNOG (current release) |
| `.X` | BayesPrism posterior Z (raw count-scale, float32 CSR) |
| `layers["z_raw"]` | copy of `.X` before normalization |

**Why raw Z?** Normalization is applied downstream in Stage 9 (`tokenize_pseudocells.py`): normalize_total to the config's `deconvolution.pseudocell_target_sum`, then the training corpus's own steps (divide by the hybrid gene medians → log2(1+x) → rank → top-k tokens, as in Stage 5). Storing raw Z preserves a faithful record of the deconvolution output and allows alternative normalization strategies in downstream analyses.

All-zero pseudo-cells (samples where BayesPrism assigns near-zero weight to a cell type) are dropped. The check below compares each tissue's pseudo-cell count with samples × cell types; a shortfall would be dropped all-zero pseudo-cells.

In [6]:
chk = inv[["tissue", "samples", "Z types", "pseudo-cells", "pc genes"]].copy()
chk["samples × types"] = chk["samples"] * chk["Z types"]
chk["dropped"] = chk["samples × types"] - chk["pseudo-cells"]
print(chk.to_string(index=False))
print(f"\nTOTAL pseudo-cells: {chk['pseudo-cells'].sum():,}; dropped all-zero pseudo-cells: {chk['dropped'].sum()}")

tissue  samples  Z types  pseudo-cells  pc genes  samples × types  dropped
   BAT       50        6           300     14162              300        0
 BLOOD       50       13           650     14920              650        0
CORTEX       50       12           600     15385              600        0
 HEART       50       16           800     15864              800        0
HIPPOC       50       11           550     15313              550        0
HYPOTH       50       13           650     14061              650        0
KIDNEY       50       15           750     14892              750        0
 LIVER       50        8           400     12766              400        0
  LUNG       50       21          1050     15233             1050        0
 OVARY       24       18           432     14176              432        0
 SKMGN       50        9           450     13142              450        0
 SKMVL       50        9           450     13141              450        0
SMLINT       50       13 

In [7]:
fracs = {t: pd.read_csv(RES_DIR / t / "estimated_fractions.csv", index_col=0) for t in TISSUES}
fig, axes = plt.subplots(4, 4, figsize=(17, 13), constrained_layout=True)
fig.suptitle("Stage 8 — BayesPrism Estimated Fractions\nDominant cell type fraction (θ) per MoTrPAC sample",
             fontsize=12, fontweight="bold")
for ax, t, color in zip(axes.flat, TISSUES, colors):
    d = fracs[t].max(axis=1).values
    ax.hist(d, bins=12, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(np.median(d), color="black", lw=1.2, ls="--", label=f"median {np.median(d):.2f}")
    ax.set_title(f"{t} (n={len(d)})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Dominant cell type fraction", fontsize=8); ax.set_ylabel("Samples", fontsize=8)
    ax.legend(fontsize=7); ax.tick_params(labelsize=7)
for ax in axes.flat[len(TISSUES):]:
    ax.axis("off")
plt.show()

In [8]:
top3 = [fracs[t].apply(lambda r: r.nlargest(3).sum(), axis=1).mean() for t in TISSUES]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle("Stage 8 — Deconvolution Concentration & Output Summary", fontsize=13, fontweight="bold")
y = np.arange(len(TISSUES))
bars = ax1.barh(y, top3, color=colors, edgecolor="white", height=0.7)
for bar, v in zip(bars, top3):
    ax1.text(v + 0.003, bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center", fontsize=8.5)
ax1.set_yticks(y); ax1.set_yticklabels(TISSUES); ax1.invert_yaxis(); ax1.set_xlim(0, 1.08)
ax1.set_xlabel("Mean top-3 cell-type fraction (per sample)")
ax1.set_title("Top-3 Cell Type Share of mRNA (θ)\n(fractions of mRNA mass, not of cells)")
ax1.axvline(np.mean(top3), color="gray", lw=1, ls="--", label=f"mean {np.mean(top3):.3f}"); ax1.legend(fontsize=8)
n_pc = inv["pseudo-cells"].values
ax2.bar(y, n_pc, color=colors, edgecolor="white", alpha=0.75, label="per tissue")
tw = ax2.twinx(); cum = np.cumsum(n_pc); tw.plot(y, cum, "k-o", ms=5, lw=1.5, label="cumulative")
for xi, cv in zip(y, cum):
    tw.annotate(f"{cv:,}", xy=(xi, cv), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=7)
tw.set_ylabel("Cumulative pseudo-cells")
ax2.set_xticks(y); ax2.set_xticklabels(TISSUES, rotation=35, ha="right"); ax2.set_ylabel("Pseudo-cells per tissue")
ax2.set_title(f"Pseudo-cell Yield (grand total: {n_pc.sum():,})")
plt.show()

## Reproducibility Notes

### Production runs

The results read here are the clean rebuild of 2026-11-02 (`pipeline/run_deconv_all.py`, one job per tissue,
12121714–12121729; `deconvolution/DECONV_REBUILD_RUNBOOK.md` §0), the default config since 2026-11-03. It reproduces,
byte for byte, the tree assembled from the earlier runs below (`deconvolution/REFS4ALL_REBUILD_RESULTS.md`). Each
run's date is in the provenance table above.

How each tissue's reference reached its current state:

| Tissues | Earlier run | Date | Reference change |
|---------|-------------|------|------------------|
| BLOOD, CORTEX, HIPPOC, KIDNEY, LIVER, LUNG, SKMGN, SKMVL, SMLINT, WATSC | array 12041931 (CORTEX rerun 12042040_3) | 2026-10-28/29 | refs4: jointly re-annotated labels (`references_v4/`, §9–§11); BLOOD whole-blood composite (§12); literature-check changes to LUNG, CORTEX, HIPPOC |
| BAT, HEART | 11318555, 11318722 | 2026-07-16 | unchanged: authors' deposited labels (BAT GSE244451, HEART SCP2828) |
| HYPOTH | 11310554–11310566 wave | 2026-07-15 | unchanged |
| SPLEEN, OVARY | array 12107800 (task 0 SPLEEN, task 1 OVARY) | 2026-11-01 | added (§14): SPLEEN GSE306764; OVARY GSE343214, females only; both read differentially only |
| TESTES | 12007286 | 2026-10-22 | OMIX767 reference with its genes lifted to the bulk's Ensembl IDs (`TESTES_OMIX767_lifted`); the July run is backed up in `data/deconvolution/_testes_rawid_backup_20260716/` |

`run.prism` wall times are in the chart above. The superseded June/July-1 runs (the earlier references) and the
July 2026 references (`references_v3/` for every tissue) are described in this notebook's git history.

### Environment and determinism

R and BayesPrism versions are in the provenance table above; Python 3.12 with `anndata`. BayesPrism was run with
its default seed and the recorded `n_cores`; the clean rebuild showed this is deterministic across runs (every
`estimated_fractions.csv` and `pred_z/` file byte-identical), so the results regenerate from the references, the
prepared bulk and the committed commands.

### Re-run Commands

```bash
# Every runnable tissue from the canonical map: validate selections -> (build) -> reference QC -> Stage 8
python pipeline/run_deconv_all.py --submit

# One tissue (the reference directory for each tissue is in deconvolution/tissue_references.yaml)
python pipeline/run_stage8.py --tissue TESTES \
    --ref-dir data/deconvolution/references_v3/TESTES_OMIX767_lifted

# Rebuild pseudo-cells only from an existing posterior Z matrix
python pipeline/run_stage8.py --tissue TESTES \
    --ref-dir data/deconvolution/references_v3/TESTES_OMIX767_lifted --from 4
```

### Data-leakage safeguard

MoTrPAC bulk RNA-seq is used only after model training. It was excluded from the Stage 5 corpus, Stage 6 prior
knowledge, and Stage 7 fine-tuning of both models.

In [9]:
import hashlib
print("=== pseudocells.h5ad inventory ===")
for _, r in inv.iterrows():
    f = (GC_DIR / r["tissue"].lower() / "pseudocells.h5ad").resolve()
    print(f"  {r['tissue']:<8} {r['pseudo-cells']:>5,} cells x {r['pc genes']:>6,} genes  "
          f"({f.stat().st_size/1e6:.0f} MB)  {f.relative_to(PROJECT.resolve())}")
print("\n=== estimated_fractions.csv checksums ===")
for t in TISSUES:
    print(f"  {t:<8} {hashlib.md5((RES_DIR / t / 'estimated_fractions.csv').read_bytes()).hexdigest()}")
print(f"\nAll reference/Z cell-type sets match: {bool(inv['types match'].all())}")

=== pseudocells.h5ad inventory ===
  BAT        300 cells x 14,162 genes  (57 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/bat/pseudocells.h5ad
  BLOOD      650 cells x 14,920 genes  (76 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/blood/pseudocells.h5ad
  CORTEX     600 cells x 15,385 genes  (100 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/cortex/pseudocells.h5ad
  HEART      800 cells x 15,864 genes  (127 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/heart/pseudocells.h5ad
  HIPPOC     550 cells x 15,313 genes  (91 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/hippoc/pseudocells.h5ad
  HYPOTH     650 cells x 14,061 genes  (101 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/hypoth/pseudocells.h5ad
  KIDNEY     750 cells x 14,892 genes  (127 MB)  data/deconvolution/genecompass_input_v2_refs4all_rebuild/kidney/pseudocells.h5ad
  LIVER      400 cells x 12,766 genes  (68 MB)  data/deconvolut